In [1]:
import pandas as pd
import random
from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# =========================
# 1. Load data
# =========================

df = pd.read_csv("../data/results.csv")
rankings = pd.read_csv("../data/fifa_mens_rank.csv")

df = df.dropna(subset=["home_score", "away_score"])
df["date"] = pd.to_datetime(df["date"])

# =========================
# 2. Create match result
# =========================

def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    else:
        return "draw"

df["result"] = df.apply(get_result, axis=1)

result_mapping = {
    "home_win": 0,
    "draw": 1,
    "away_win": 2
}

df["result_numeric"] = df["result"].map(result_mapping)

# =========================
# 3. Encode teams
# =========================

all_teams = pd.concat([df["home_team"], df["away_team"]]).unique()

team_encoder = LabelEncoder()
team_encoder.fit(all_teams)

df["home_team_encoded"] = team_encoder.transform(df["home_team"])
df["away_team_encoded"] = team_encoder.transform(df["away_team"])

# =========================
# 4. Team statistics from recent matches
# =========================

recent_df = df[df["date"].dt.year >= 2018]

home_stats = recent_df.groupby("home_team").agg({
    "home_score": "mean",
    "away_score": "mean"
}).rename(columns={
    "home_score": "avg_goals_scored_home",
    "away_score": "avg_goals_conceded_home"
})

away_stats = recent_df.groupby("away_team").agg({
    "away_score": "mean",
    "home_score": "mean"
}).rename(columns={
    "away_score": "avg_goals_scored_away",
    "home_score": "avg_goals_conceded_away"
})

# =========================
# 5. FIFA rankings
# =========================

current_rankings = rankings.sort_values(
    by=["date", "semester"],
    ascending=False
).drop_duplicates(subset=["team"], keep="first")

rank_dict = dict(zip(current_rankings["team"], current_rankings["rank"]))

# =========================
# 6. Add features to dataframe
# =========================

df = df.merge(home_stats, left_on="home_team", right_index=True, how="left")
df = df.merge(away_stats, left_on="away_team", right_index=True, how="left")

df["home_rank"] = df["home_team"].map(rank_dict)
df["away_rank"] = df["away_team"].map(rank_dict)

df = df.dropna()

df["rank_difference"] = df["away_rank"] - df["home_rank"]

# =========================
# 7. Train model
# =========================

features = [
    "home_team_encoded",
    "away_team_encoded",
    "neutral",
    "avg_goals_scored_home",
    "avg_goals_conceded_home",
    "avg_goals_scored_away",
    "avg_goals_conceded_away",
    "home_rank",
    "away_rank",
    "rank_difference"
]

X = df[features]
y = df["result_numeric"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))

print("Model accuracy:", accuracy)

# =========================
# 8. Match prediction function
# =========================

ranking_name_mapping = {
    "United States": "USA",
    "Iran": "IR Iran",
    "South Korea": "Korea Republic",
    "Cape Verde": "Cape Verde Islands",
    "DR Congo": "Congo DR",
    "Ivory Coast": "Côte d'Ivoire"
}

def get_rank(team):
    ranking_name = ranking_name_mapping.get(team, team)
    return rank_dict.get(ranking_name)

def predict_match(home_team, away_team, neutral=True):

    if home_team not in home_stats.index:
        return None

    if away_team not in away_stats.index:
        return None

    if home_team not in team_encoder.classes_:
        return None

    if away_team not in team_encoder.classes_:
        return None

    home_rank = get_rank(home_team)
    away_rank = get_rank(away_team)

    if home_rank is None or away_rank is None:
        return None

    home_features = home_stats.loc[home_team]
    away_features = away_stats.loc[away_team]

    X_match = pd.DataFrame([{
        "home_team_encoded": team_encoder.transform([home_team])[0],
        "away_team_encoded": team_encoder.transform([away_team])[0],
        "neutral": neutral,
        "avg_goals_scored_home": home_features["avg_goals_scored_home"],
        "avg_goals_conceded_home": home_features["avg_goals_conceded_home"],
        "avg_goals_scored_away": away_features["avg_goals_scored_away"],
        "avg_goals_conceded_away": away_features["avg_goals_conceded_away"],
        "home_rank": home_rank,
        "away_rank": away_rank,
        "rank_difference": away_rank - home_rank
    }])

    probs = model.predict_proba(X_match)[0]

    return {
        "home_win": probs[0],
        "draw": probs[1],
        "away_win": probs[2]
    }

print("France vs Brazil:", predict_match("France", "Brazil"))

# =========================
# 9. Tournament simulation
# =========================

def simulate_match(team1, team2):
    probs = predict_match(team1, team2, neutral=True)

    if probs is None:
        return random.choice([team1, team2])

    outcomes = [team1, "draw", team2]
    weights = [
        probs["home_win"],
        probs["draw"],
        probs["away_win"]
    ]

    result = random.choices(outcomes, weights=weights, k=1)[0]

    if result == "draw":
        return random.choice([team1, team2])

    return result

# =========================
# 10. 2026 World Cup teams
# =========================

worldcup_2026_teams = [
    "Canada", "Mexico", "United States",
    "Australia", "Iraq", "Iran", "Japan", "Jordan", "South Korea", "Qatar", "Saudi Arabia", "Uzbekistan",
    "Algeria", "Cape Verde", "DR Congo", "Ivory Coast", "Egypt", "Ghana", "Morocco", "Senegal", "South Africa", "Tunisia",
    "Curaçao", "Haiti", "Panama",
    "Argentina", "Brazil", "Colombia", "Ecuador", "Paraguay", "Uruguay",
    "New Zealand",
    "Austria", "Belgium", "Bosnia and Herzegovina", "Croatia", "Czech Republic", "England", "France", "Germany", "Netherlands",
    "Norway", "Portugal", "Scotland", "Spain", "Sweden", "Switzerland", "Turkey"
]

print("Number of teams:", len(worldcup_2026_teams))

# =========================
# 11. 48-team tournament simulation
# =========================

def simulate_worldcup_48(teams):
    teams = teams.copy()

    # Sort by FIFA rank
    teams = sorted(teams, key=lambda team: get_rank(team))

    # Top 16 teams get a bye
    top_16 = teams[:16]
    remaining_32 = teams[16:]

    random.shuffle(remaining_32)

    round_of_32_winners = []

    for i in range(0, len(remaining_32), 2):
        winner = simulate_match(remaining_32[i], remaining_32[i + 1])
        round_of_32_winners.append(winner)

    round_of_32 = top_16 + round_of_32_winners
    random.shuffle(round_of_32)

    while len(round_of_32) > 1:
        next_round = []

        for i in range(0, len(round_of_32), 2):
            winner = simulate_match(round_of_32[i], round_of_32[i + 1])
            next_round.append(winner)

        round_of_32 = next_round

    return round_of_32[0]

# =========================
# 12. Run simulations
# =========================

simulations = 10000
champions = []

for i in range(simulations):
    champion = simulate_worldcup_48(worldcup_2026_teams)
    champions.append(champion)

winner_counts = Counter(champions)

worldcup_probabilities = pd.DataFrame({
    "team": winner_counts.keys(),
    "wins": winner_counts.values()
})

worldcup_probabilities["winning_percentage"] = (
    worldcup_probabilities["wins"] / simulations * 100
)

# Add teams with 0 wins
all_teams_df = pd.DataFrame({"team": worldcup_2026_teams})

worldcup_probabilities = all_teams_df.merge(
    worldcup_probabilities,
    on="team",
    how="left"
)

worldcup_probabilities["wins"] = worldcup_probabilities["wins"].fillna(0).astype(int)
worldcup_probabilities["winning_percentage"] = worldcup_probabilities["winning_percentage"].fillna(0)

worldcup_probabilities = worldcup_probabilities.sort_values(
    by="winning_percentage",
    ascending=False
)

worldcup_probabilities

Model accuracy: 0.5271684945164506
France vs Brazil: {'home_win': np.float64(0.780892857142857), 'draw': np.float64(0.06984523809523811), 'away_win': np.float64(0.14926190476190476)}
Number of teams: 48


,team,wins,winning_percentage
26,Brazil,1343,13.43
25,Argentina,1081,10.81
38,France,1057,10.57
44,Spain,895,8.95
39,Germany,805,8.05
40,Netherlands,698,6.98
30,Uruguay,538,5.38
42,Portugal,479,4.79
37,England,365,3.65
33,Belgium,326,3.26
